## Multimodal AI Sidekick

A self-contained Sidekick application with a Gradio UI, specifically tailored for creating world-class multimodal AI systems (text, vision, audio).
It connects LangGraph StateGraph, Evaluator-Worker structure, and specialized Async Tools (Playwright, Python REPL, File Management, Google Search, and Wikipedia).

In [ ]:
import os
import uuid
import asyncio
import nest_asyncio
import requests
from datetime import datetime
from typing import Annotated, List, Any, Optional, Dict
from typing_extensions import TypedDict

import gradio as gr
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# LangChain / LangGraph imports
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain.agents import Tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# Tools imports
from playwright.async_api import async_playwright
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit, FileManagementToolkit
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_experimental.tools import PythonREPLTool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper

nest_asyncio.apply()
load_dotenv(override=True)

In [ ]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool

class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(
        description="True if more input is needed from the user, or clarifications, or the assistant is stuck"
    )

In [ ]:
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"

serper = GoogleSerperAPIWrapper()

async def playwright_tools():
    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(headless=False)
    toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=browser)
    return toolkit.get_tools(), browser, playwright

def push(text: str):
    """Send a push notification to the user"""
    if not pushover_token or not pushover_user:
        return "Pushover API tokens not configured. Notification skipped."
    requests.post(pushover_url, data={"token": pushover_token, "user": pushover_user, "message": text})
    return "success"

def get_file_tools():
    os.makedirs("sandbox", exist_ok=True)
    toolkit = FileManagementToolkit(root_dir="sandbox")
    return toolkit.get_tools()

async def other_tools():
    push_tool = Tool(name="send_push_notification", func=push, description="Use this tool when you want to send a push notification")
    file_tools = get_file_tools()
    tool_search = Tool(name="search", func=serper.run, description="Use this tool when you want to get the results of an online web search")
    wikipedia = WikipediaAPIWrapper()
    wiki_tool = WikipediaQueryRun(api_wrapper=wikipedia)
    python_repl = PythonREPLTool()
    return file_tools + [push_tool, tool_search, python_repl, wiki_tool]

In [ ]:
class Sidekick:
    def __init__(self):
        self.worker_llm_with_tools = None
        self.evaluator_llm_with_output = None
        self.tools = None
        self.graph = None
        self.sidekick_id = str(uuid.uuid4())
        self.memory = MemorySaver()
        self.browser = None
        self.playwright = None

    async def setup(self):
        pl_tools, self.browser, self.playwright = await playwright_tools()
        self.tools = pl_tools + await other_tools()
        worker_llm = ChatOpenAI(model="gpt-4o-mini")
        self.worker_llm_with_tools = worker_llm.bind_tools(self.tools)
        evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
        self.evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)
        await self.build_graph()

    def worker(self, state: State) -> Dict[str, Any]:
        system_message = f"""You are an elite, world-class Multimodal AI Systems sidekick assistant.
    Your primary objective is to help the user design, orchestrate, and perfect intelligent systems that integrate text, vision, and audio natively.
    You keep working on a task using your massive toolset until you have a clarifying question, or the success criteria is met.
    Your tools include browser navigation, python code execution (use print() for output), sandbox file management, Wikipedia, and Google Search.
    The current date and time is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}.

    This is the success criteria:
    {state["success_criteria"]}

    Reply with a question for the user OR the final technical response.
    Example question: "Question: Do you want this built for streaming OpenAI Realtime API audio, or offline whisper generation?"
    """

        if state.get("feedback_on_work"):
            system_message += f"""\n
    Previously your reply was evaluated and deemed incomplete. Feedback:
    {state["feedback_on_work"]}
    Please continue the assignment taking this feedback into account to properly meet the success criteria."""

        found_system = False
        messages = state["messages"]
        for message in messages:
            if isinstance(message, SystemMessage):
                message.content = system_message
                found_system = True
        if not found_system:
            messages = [SystemMessage(content=system_message)] + messages

        response = self.worker_llm_with_tools.invoke(messages)
        return {"messages": [response]}

    def worker_router(self, state: State) -> str:
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools"
        return "evaluator"

    def format_conversation(self, messages: List[Any]) -> str:
        conversation = "Conversation history:\n\n"
        for message in messages:
            if isinstance(message, HumanMessage):
                conversation += f"User: {message.content}\n"
            elif isinstance(message, AIMessage):
                text = message.content or "[Tools use]"
                conversation += f"Assistant: {text}\n"
        return conversation

    def evaluator(self, state: State) -> State:
        last_response = state["messages"][-1].content
        system_message = "You are an evaluator determining if an AI task has been completed successfully based on success criteria."
        user_message = f"""Evaluate the conversation. Decide what action to take based on the final Assistant response.

    Conversation:
    {self.format_conversation(state["messages"])}

    Success criteria:
    {state["success_criteria"]}

    Final response:
    {last_response}

    Provide feedback, decide if success criteria is completely met, and if user input is needed.
    Remember, give the Assistant the benefit of the doubt if they say they used their Python tools/files to verify the result.
    """
        if state.get("feedback_on_work"):
            user_message += f"\nPrior feedback: {state['feedback_on_work']}\nIf you're seeing repeated mistakes, require user input."

        evaluator_messages = [SystemMessage(content=system_message), HumanMessage(content=user_message)]
        eval_result = self.evaluator_llm_with_output.invoke(evaluator_messages)

        return {
            "messages": [{"role": "assistant", "content": f"Evaluator Feedback: {eval_result.feedback}"}],
            "feedback_on_work": eval_result.feedback,
            "success_criteria_met": eval_result.success_criteria_met,
            "user_input_needed": eval_result.user_input_needed,
        }

    def route_based_on_evaluation(self, state: State) -> str:
        if state["success_criteria_met"] or state["user_input_needed"]:
            return "END"
        return "worker"

    async def build_graph(self):
        graph_builder = StateGraph(State)
        graph_builder.add_node("worker", self.worker)
        graph_builder.add_node("tools", ToolNode(tools=self.tools))
        graph_builder.add_node("evaluator", self.evaluator)
        graph_builder.add_conditional_edges("worker", self.worker_router, {"tools": "tools", "evaluator": "evaluator"})
        graph_builder.add_edge("tools", "worker")
        graph_builder.add_conditional_edges("evaluator", self.route_based_on_evaluation, {"worker": "worker", "END": END})
        graph_builder.add_edge(START, "worker")
        self.graph = graph_builder.compile(checkpointer=self.memory)

    async def run_superstep(self, message, success_criteria, history):
        config = {"configurable": {"thread_id": self.sidekick_id}}
        state = {
            "messages": message,
            "success_criteria": success_criteria or "Output should be clear, concise, and technically accurate regarding multimodal AI.",
            "feedback_on_work": None,
            "success_criteria_met": False,
            "user_input_needed": False,
        }
        result = await self.graph.ainvoke(state, config=config)
        user = {"role": "user", "content": message}
        reply = {"role": "assistant", "content": result["messages"][-2].content}
        feedback = {"role": "assistant", "content": result["messages"][-1].content}
        return history + [user, reply, feedback]

    def cleanup(self):
        if self.browser:
            try:
                loop = asyncio.get_running_loop()
                loop.create_task(self.browser.close())
                if self.playwright:
                    loop.create_task(self.playwright.stop())
            except RuntimeError:
                asyncio.run(self.browser.close())
                if self.playwright:
                    asyncio.run(self.playwright.stop())

In [ ]:
async def initialize_sidekick():
    sk = Sidekick()
    await sk.setup()
    return sk

async def process_message(state_sidekick, message, criteria, history):
    results = await state_sidekick.run_superstep(message, criteria, history)
    return results, state_sidekick

async def ui_reset():
    new_sidekick = Sidekick()
    await new_sidekick.setup()
    return "", "", None, new_sidekick

def free_resources(state_sidekick):
    if state_sidekick:
        state_sidekick.cleanup()

with gr.Blocks(theme=gr.themes.Default(primary_hue="emerald")) as ui:
    gr.Markdown("# 🚀 Multimodal AI Sidekick Workflow")
    gr.Markdown("Your elite personal co-worker dedicated to architecting and deploying multimodal AI systems. Equip it with success criteria and watch it build!")
    
    sidekick_state = gr.State(delete_callback=free_resources)

    with gr.Row():
        chatbot = gr.Chatbot(label="Multimodal Agent", height=350, type="messages")
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(show_label=False, placeholder="Your multimodal systems request to the Sidekick")
        with gr.Row():
            success_criteria = gr.Textbox(show_label=False, placeholder="What are your concrete success criteria for this execution?")
    with gr.Row():
        reset_button = gr.Button("Reset System", variant="stop")
        go_button = gr.Button("Execute", variant="primary")

    ui.load(initialize_sidekick, [], [sidekick_state])
    message.submit(process_message, [sidekick_state, message, success_criteria, chatbot], [chatbot, sidekick_state])
    success_criteria.submit(process_message, [sidekick_state, message, success_criteria, chatbot], [chatbot, sidekick_state])
    go_button.click(process_message, [sidekick_state, message, success_criteria, chatbot], [chatbot, sidekick_state])
    reset_button.click(ui_reset, [], [message, success_criteria, chatbot, sidekick_state])

ui.launch(inbrowser=True)